# Gym99 Dual-GPU Training — Kaggle T4×2
Runs two experiments in parallel, each pinned to one GPU. Live progress plot updates every minute.

Edit **Cell 1** only, then Run All.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1 — CONFIG
# ═══════════════════════════════════════════════════════════════

# ── Paths ─────────────────────────────────────────────────────
REPO_DIR   = '/kaggle/working/Yolo-ST-GCN'
BRANCH     = 'refactor-1'
GYM288_DIR = '/kaggle/working/Gym288-skeleton'
GYM288_PKL = '/kaggle/working/Gym288-skeleton/gym_288_skeleton.pkl'
GYM99_PKL  = '/kaggle/working/gym99_from_gym288.pkl'

# ── Shared flags (applied to both experiments) ────────────────
BASE_FLAGS = [
    '--auto_build_from_gym288',
    '--gym288_dataset_path', GYM288_PKL,
    '--dataset_path',        GYM99_PKL,
    '--epochs',              '90',
    '--batch_size',          '64',
    '--lr',                  '0.001',
    '--weight_decay',        '1e-4',
    '--num_workers',         '0',
    '--joint_spec_name',     'coco18',
    '--use_two_stream',
    '--loss_name',           'focal',
    '--focal_alpha_mode',    'sqrt_inverse',
    '--save_every_epochs',   '10',
    '--train_data_mode',     'preload_vram',
]

# ── Experiment 1 (GPU 0) ──────────────────────────────────────
EXP1_NAME  = 'bbox_norm+warmup'
EXP1_FLAGS = ['--bbox_norm', '--warmup_epochs', '10']

# ── Experiment 2 (GPU 1) ──────────────────────────────────────
EXP2_NAME  = 'baseline'
EXP2_FLAGS = ['--warmup_epochs', '0']   # no bbox_norm, no warmup

# ── Plot refresh interval (seconds) ──────────────────────────
POLL_INTERVAL = 60

print(f'Exp1 [{EXP1_NAME}]: {EXP1_FLAGS}')
print(f'Exp2 [{EXP2_NAME}]: {EXP2_FLAGS}')

In [ ]:
# ── Cell 2: Environment setup ─────────────────────────────────
import os, sys, subprocess, time, json, re
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Repo ready:', os.getcwd())

In [ ]:
# ── Cell 3: Download Gym288 ───────────────────────────────────
from huggingface_hub import snapshot_download

dl = Path(GYM288_DIR)
if dl.exists() and any(dl.rglob('*.pkl')):
    print('Gym288 already present.')
else:
    print('Downloading Gym288-skeleton...')
    dl.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id='Lozumi/Gym288-skeleton', repo_type='dataset',
                      local_dir=str(dl), local_dir_use_symlinks=False)

pkl_candidates = sorted(dl.rglob('*.pkl'))
if not pkl_candidates:
    raise FileNotFoundError(f'No .pkl found under {GYM288_DIR}')
GYM288_PKL = str(pkl_candidates[0])

# Patch BASE_FLAGS with resolved path
idx = BASE_FLAGS.index('--gym288_dataset_path') + 1
BASE_FLAGS[idx] = GYM288_PKL
print('Gym288 pickle:', GYM288_PKL)

In [ ]:
# ── Cell 4: GPU check ─────────────────────────────────────────
import torch
n = torch.cuda.device_count()
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count     : {n}')
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB')
if n < 2:
    print('[warning] Only 1 GPU found — both experiments will run on GPU 0 sequentially.')

In [ ]:
# ── Cell 5: Launch both experiments ──────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import clear_output

OUT1 = '/kaggle/working/outputs/' + EXP1_NAME.replace(' ', '_')
OUT2 = '/kaggle/working/outputs/' + EXP2_NAME.replace(' ', '_')
Path(OUT1).mkdir(parents=True, exist_ok=True)
Path(OUT2).mkdir(parents=True, exist_ok=True)

LOG1 = Path(OUT1) / 'train.log'
LOG2 = Path(OUT2) / 'train.log'

def build_cmd(out_dir, extra_flags):
    return [sys.executable, 'scripts/train_gym99.py'] + BASE_FLAGS + ['--out_dir', out_dir] + extra_flags

cmd1 = build_cmd(OUT1, EXP1_FLAGS)
cmd2 = build_cmd(OUT2, EXP2_FLAGS)

print(f'Launching [{EXP1_NAME}] on GPU 0 ...')
print(f'Launching [{EXP2_NAME}] on GPU 1 ...')

env1 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
env2 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '1' if torch.cuda.device_count() > 1 else '0'}

p1 = subprocess.Popen(cmd1, env=env1, stdout=open(LOG1, 'w'), stderr=subprocess.STDOUT)
p2 = subprocess.Popen(cmd2, env=env2, stdout=open(LOG2, 'w'), stderr=subprocess.STDOUT)

# ── Live monitoring loop ───────────────────────────────────────
# Parses epoch lines from log files; plots every POLL_INTERVAL seconds.
# Pattern: "Epoch X/Y  train_loss=Z  train_acc=Z  val_loss=Z  val_acc=Z  val_f1=Z"

EPOCH_RE = re.compile(
    r'Epoch\s+(\d+)/\d+\s+'
    r'train_loss=([\d.]+)\s+train_acc=([\d.]+)\s+'
    r'val_loss=([\d.]+)\s+val_acc=([\d.]+)\s+val_f1=([\d.]+)'
)

def parse_log(log_path):
    h = {'epoch': [], 'train_loss': [], 'val_loss': [],
         'train_acc': [], 'val_acc': [], 'val_f1': []}
    try:
        for line in Path(log_path).read_text().splitlines():
            m = EPOCH_RE.search(line)
            if m:
                h['epoch'].append(int(m.group(1)))
                h['train_loss'].append(float(m.group(2)))
                h['train_acc'].append(float(m.group(3)))
                h['val_loss'].append(float(m.group(4)))
                h['val_acc'].append(float(m.group(5)))
                h['val_f1'].append(float(m.group(6)))
    except FileNotFoundError:
        pass
    return h

def plot_progress(h1, h2, elapsed_sec):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = [
        ('train_loss', 'val_loss',  'Loss'),
        ('train_acc',  'val_acc',   'Accuracy'),
        ('val_f1',     None,        'Val Macro-F1'),
    ]
    colors = {'exp1': ('#4878cf', '#4878cf'), 'exp2': ('#d65f5f', '#d65f5f')}

    for ax, (tr_key, va_key, title) in zip(axes, metrics):
        for h, name, (c1, c2) in [(h1, EXP1_NAME, colors['exp1']),
                                   (h2, EXP2_NAME, colors['exp2'])]:
            ep = h['epoch']
            if not ep:
                continue
            ax.plot(ep, h[tr_key], color=c1, linewidth=1.8,
                    label=f'{name} train', linestyle='-')
            if va_key:
                ax.plot(ep, h[va_key], color=c2, linewidth=1.8,
                        label=f'{name} val', linestyle='--')
            else:
                ax.plot(ep, h[tr_key], color=c1, linewidth=1.8, label=name)
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    h, m = divmod(int(elapsed_sec), 3600)
    m, s = divmod(m, 60)
    ep1 = h1['epoch'][-1] if h1['epoch'] else 0
    ep2 = h2['epoch'][-1] if h2['epoch'] else 0
    fig.suptitle(
        f'Elapsed {h}h{m:02d}m  |  '
        f'{EXP1_NAME}: ep {ep1}/90  |  {EXP2_NAME}: ep {ep2}/90',
        fontsize=10
    )
    plt.tight_layout()
    plt.savefig('/kaggle/working/outputs/progress.png', dpi=100, bbox_inches='tight')
    plt.show()

t0 = time.time()
print(f'Both processes launched. Polling every {POLL_INTERVAL}s ...\n')

while p1.poll() is None or p2.poll() is None:
    time.sleep(POLL_INTERVAL)
    h1 = parse_log(LOG1)
    h2 = parse_log(LOG2)
    clear_output(wait=True)
    plot_progress(h1, h2, time.time() - t0)
    status1 = 'running' if p1.poll() is None else f'done (exit {p1.returncode})'
    status2 = 'running' if p2.poll() is None else f'done (exit {p2.returncode})'
    print(f'[{EXP1_NAME}] {status1}')
    print(f'[{EXP2_NAME}] {status2}')

elapsed = time.time() - t0
h, m = divmod(int(elapsed), 3600)
m, s = divmod(m, 60)
print(f'\nBoth experiments finished in {h}h {m}m {s}s')
print(f'Exit codes: exp1={p1.returncode}  exp2={p2.returncode}')

In [ ]:
# ── Cell 6: Final comparison ──────────────────────────────────
import matplotlib.pyplot as plt

with open(Path(OUT1) / 'history.json') as f: hist1 = json.load(f)
with open(Path(OUT2) / 'history.json') as f: hist2 = json.load(f)
with open(Path(OUT1) / 'metrics_train_gym99.json') as f: m1 = json.load(f)
with open(Path(OUT2) / 'metrics_train_gym99.json') as f: m2 = json.load(f)

print(f'=== {EXP1_NAME} ===')
for k, v in m1.items(): print(f'  {k}: {v}')
print(f'\n=== {EXP2_NAME} ===')
for k, v in m2.items(): print(f'  {k}: {v}')

epochs_x1 = range(1, len(hist1['train_loss']) + 1)
epochs_x2 = range(1, len(hist2['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_pairs = [
    ('train_loss', 'val_loss',  'Loss'),
    ('train_acc',  'val_acc',   'Accuracy'),
    ('val_f1',     None,        'Val Macro-F1'),
]

for ax, (tr_key, va_key, title) in zip(axes, plot_pairs):
    ax.plot(epochs_x1, hist1[tr_key], color='#4878cf', label=f'{EXP1_NAME} train')
    ax.plot(epochs_x2, hist2[tr_key], color='#d65f5f', label=f'{EXP2_NAME} train')
    if va_key:
        ax.plot(epochs_x1, hist1[va_key], color='#4878cf', linestyle='--', label=f'{EXP1_NAME} val')
        ax.plot(epochs_x2, hist2[va_key], color='#d65f5f', linestyle='--', label=f'{EXP2_NAME} val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

fig.suptitle(f'{EXP1_NAME} vs {EXP2_NAME}  —  90 epochs', fontsize=11)
plt.tight_layout()
fig.savefig('/kaggle/working/outputs/final_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved final_comparison.png')